In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import ListedColormap
import seaborn as sns
import ast
import math
import scienceplots

plt.style.use("default")
plt.style.use(["science", "grid"])

In [ ]:
Images_Path = Path("../Images")

Iteration_Plots_Path = Path(Images_Path / 'Iteration_PyPlots/')
Projection_Plots_Path = Path(Images_Path / 'Projection_PyPlots/')

Empty_Plots_Path = Path(Iteration_Plots_Path / "Empty/")
Full_Plots_Path = Path(Iteration_Plots_Path / "Full/")

Empty_Plots_Path.mkdir(parents=True, exist_ok=True)
Full_Plots_Path.mkdir(parents=True, exist_ok=True)
Projection_Plots_Path.mkdir(parents=True, exist_ok=True)

In [ ]:
data = pd.read_csv("../results.csv")
data.info()

In [ ]:
data['x_d'].apply(tuple)

def str_to_arr(x : str) -> np.array:
  return np.fromstring(x[1:-1], sep=',')

def str_to_list(x : str) -> list:
  return str_to_arr(x).tolist()

def str_to_tuple(x : str) -> tuple:
  return tuple(str_to_arr(x))

data['final_position'] = data['final_position'].apply(str_to_tuple)
data['x0'] = data['x0'].apply(str_to_tuple)
data['x_d'] = data['x_d'].apply(str_to_tuple)

data['position_over_time'] = data['position_over_time'].apply(
  lambda x: np.nan if type(x) != str and math.isnan(x) else np.array(ast.literal_eval(x))
)

data.info()
data.head()

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

groups = data.groupby(by=["method", "k_spring", "x_d"])
iter_color = LinearSegmentedColormap.from_list(
    "iter_smooth",
    ["#1E9600", "#FFF200", "#FF0000"]
)

for (method, k_spring, x_d) ,g in groups:
    fig = plt.figure(figsize=(10, 6))

    plt.hlines(y = 0.0, xmin=-2, xmax=2, linestyles='dashed')
    plt.vlines(x = 0.0, ymin = 0, ymax = max(x_d[1], 2.0), linestyles='dashed')
    plt.scatter(x_d[0], x_d[1], marker='*', c='purple', label=f"Desired Position [{x_d[0]}, {x_d[1]}]", sizes=np.array([60]))

    X, Y, C = [], [], []
    for index, row in g.iterrows():
      x0 = row.loc['x0']
      final_position = row.loc['final_position']
      iteration = row.loc['iterations']

      if iteration == -1:
        plt.scatter(x0[0], x0[1], c='k')
      else:
        x, y = x0[0], x0[1]
        X.append(x0[0])
        Y.append(x0[1])
        C.append(iteration)

        dx, dy = final_position[0] - x, final_position[1] - y
        plt.arrow(x, y, dx, dy, width=0.025)


    sc = plt.scatter(X, Y, c=C, cmap = iter_color, vmin = 0, vmax = 30, sizes=np.full(len(X), 80))
    plt.colorbar(sc, label="Iterations")
    plt.title(f"{method}\nk={k_spring}", fontsize="large")
    legend = plt.legend(fancybox=False, edgecolor="black")
    legend.get_frame().set_linewidth(0.5)

    plt.savefig((Empty_Plots_Path if len(X) == 0 else Full_Plots_Path)  / f"IterationPlot{method}_k={k_spring}_h{x_d[1]}.pdf", dpi=300, format="pdf")
    plt.close()



In [ ]:
converged_entries = data[data['position_over_time'].notna()]
for row in converged_entries.itertuples(index=False):
  position_over_time = np.array(row.position_over_time)
  x_d = np.array(row.x_d)
  x0 = np.array(row.x0)

  x_d_str = ", ".join(f"{v:.2f}" for v in x_d)
  x0_str = ", ".join(f"{v:.2f}" for v in x0)

  x = position_over_time[:, 0]
  y = position_over_time[:, 1]
  dx = np.diff(x)
  dy = np.diff(y)

  allX = np.hstack((x, x_d[0], [0]))
  allY = np.hstack((y, x_d[1], [0]))

  minX, maxX = allX.min(), allX.max()
  minY, maxY = allY.min(), allY.max()

  speed = np.sqrt(dx**2 + dy**2)

  plt.figure(figsize=(10, 6))
  plt.hlines(y = 0.0, xmin=minX, xmax=maxX, linestyles='dashed')
  plt.vlines(x = 0.0, ymin=minY, ymax=maxY, linestyles='dashed')

  Q = plt.quiver(x[:-1],y[:-1], dx, dy, speed, angles='xy', scale_units='xy', scale=1, cmap='viridis')
  plt.colorbar(mappable=Q, label=r"Step magnitude $| \Delta x \|$")

  plt.scatter([x[0]], [y[0]], marker='o', c='lime', label=f"Starting Position [{x0_str}]")
  plt.scatter(x_d[0], x_d[1], marker='*', c='purple', label=f"Desired Position[{x_d_str}]")


  ax = plt.gca()
  ax.text(
    0.02, 0.98,
    f"Method : {row.method}\nm = {row.mass}\nk = {row.k_spring}\nN = {row.N}\nalpha = {row.alpha}",
    fontsize=10,
    ha='left',
    va='top',
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
    transform=ax.transAxes
  )

  plt.legend()
  plt.xlabel("X")
  plt.ylabel("Y")

  plt.savefig(Projection_Plots_Path / f"Projection_N{row.N}_alpha{row.alpha}_k{row.k_spring}_m{row.mass}_x0[{x0_str}]_h{x_d[1]}_{row.method}.pdf", dpi=300, format="pdf")
  plt.close()



In [ ]:
SE1_convergence = converged_entries[converged_entries["method"].str.contains("SE1")]

for group_key, group_df in SE1_convergence.groupby(by=['alpha', 'N', 'k_spring', 'x0', 'x_d', 'mass']):
  modified_method_series = group_df[group_df["method"].str.contains("Modified")]
  non_modified_method_series = group_df[~group_df["method"].str.contains("Modified")]

  if modified_method_series.empty or non_modified_method_series.empty:
    continue

  fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(20, 6), sharex=True, sharey=True)

  alpha, N, k, x0, x_d, m = group_key
  x0 = np.array(x0)
  x_d = np.array(x_d)

  x_d_str = ", ".join(f"{v:.2f}" for v in x_d)
  x0_str = ", ".join(f"{v:.2f}" for v in x0)


  position_over_time = non_modified_method_series['position_over_time'].iloc[0]
  modified_position_over_time = modified_method_series['position_over_time'].iloc[0]

  x = position_over_time[:, 0]
  y = position_over_time[:, 1]
  dx = np.diff(x)
  dy = np.diff(y)
  speed = np.sqrt(dx**2 + dy**2)

  x_modified = modified_position_over_time[:, 0]
  y_modified = modified_position_over_time[:, 1]
  dx_modified = np.diff(x_modified)
  dy_modified = np.diff(y_modified)
  speed_modified = np.sqrt(dx_modified**2 + dy_modified**2)

  allX = np.hstack((x, x_modified, x_d[0], [0]))
  allY = np.hstack((y, y_modified, x_d[1], [0]))

  minX, maxX = allX.min(), allX.max()
  minY, maxY = allY.min(), allY.max()

  for i in [0, 1]:
    axs[i].hlines(y = 0.0, xmin=minX, xmax=maxX, linestyles='dashed')
    axs[i].vlines(x = 0.0, ymin=minY, ymax=maxY, linestyles='dashed')
    axs[i].set_xlabel("X")
    axs[i].set_ylabel("Y")
    axs[i].scatter(x0[0], x0[1], marker='o', c='lime', label=f"Starting Position[{x0_str}]")
    axs[i].scatter(x_d[0], x_d[1], marker='*', c='purple', label=f"Desired Position[{x_d_str}]")
    axs[i].text(
    0.02, 0.98,
    f"Method : {"Modified SE1" if i == 1 else "SE1"}\nm = {m}\nk = {k}\nN = {N}\nalpha = {alpha}",
    fontsize=10,
    ha='left',
    va='top',
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
    transform=axs[i].transAxes
    )
    axs[i].legend()

  Q = axs[0].quiver(
    x[:-1], y[:-1],
    dx, dy,
    speed,
    angles='xy',
    scale_units='xy',
    scale=1,
    cmap='viridis',
  )

  Q_modified = axs[1].quiver(
    x_modified[:-1], y_modified[:-1],
    dx_modified, dy_modified,
    speed_modified,
    angles='xy',
    scale_units='xy',
    scale=1,
    cmap='viridis',
  )

  plt.colorbar(mappable=Q, label=r"Step magnitude $| \Delta x \|$")
  plt.colorbar(mappable=Q_modified, label=r"Step magnitude $| \Delta x \|$")



In [ ]:
SE2_convergence = converged_entries[converged_entries["method"].str.contains("SE2")]

for group_key, group_df in SE2_convergence.groupby(by=['alpha', 'N', 'k_spring', 'x0', 'x_d', 'mass']):
  modified_method_series = group_df[group_df["method"].str.contains("Modified")]
  non_modified_method_series = group_df[~group_df["method"].str.contains("Modified")]

  if modified_method_series.empty or non_modified_method_series.empty:
    continue

  fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(20, 6), sharex=True, sharey=True)

  alpha, N, k, x0, x_d, m = group_key
  x0 = np.array(x0)
  x_d = np.array(x_d)

  x_d_str = ", ".join(f"{v:.2f}" for v in x_d)
  x0_str = ", ".join(f"{v:.2f}" for v in x0)


  position_over_time = non_modified_method_series['position_over_time'].iloc[0]
  modified_position_over_time = modified_method_series['position_over_time'].iloc[0]

  x = position_over_time[:, 0]
  y = position_over_time[:, 1]
  dx = np.diff(x)
  dy = np.diff(y)
  speed = np.sqrt(dx**2 + dy**2)

  x_modified = modified_position_over_time[:, 0]
  y_modified = modified_position_over_time[:, 1]
  dx_modified = np.diff(x_modified)
  dy_modified = np.diff(y_modified)
  speed_modified = np.sqrt(dx_modified**2 + dy_modified**2)

  allX = np.hstack((x, x_modified, x_d[0], [0]))
  allY = np.hstack((y, y_modified, x_d[1], [0]))

  minX, maxX = allX.min(), allX.max()
  minY, maxY = allY.min(), allY.max()

  for i in [0, 1]:
    axs[i].hlines(y = 0.0, xmin=minX, xmax=maxX, linestyles='dashed')
    axs[i].vlines(x = 0.0, ymin=minY, ymax=maxY, linestyles='dashed')
    axs[i].set_xlabel("X")
    axs[i].set_ylabel("Y")
    axs[i].scatter(x0[0], x0[1], marker='o', c='lime', label=f"Starting Position[{x0_str}]")
    axs[i].scatter(x_d[0], x_d[1], marker='*', c='purple', label=f"Desired Position[{x_d_str}]")
    axs[i].text(
    0.02, 0.98,
    f"Method : {"Modified SE2" if i == 1 else "SE2"}\nm = {m}\nk = {k}\nN = {N}\nalpha = {alpha}",
    fontsize=10,
    ha='left',
    va='top',
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
    transform=axs[i].transAxes
    )
    axs[i].legend()

  Q = axs[0].quiver(
    x[:-1], y[:-1],
    dx, dy,
    speed,
    angles='xy',
    scale_units='xy',
    scale=1,
    cmap='viridis',
  )

  Q_modified = axs[1].quiver(
    x_modified[:-1], y_modified[:-1],
    dx_modified, dy_modified,
    speed_modified,
    angles='xy',
    scale_units='xy',
    scale=1,
    cmap='viridis',
  )

  plt.colorbar(mappable=Q, label=r"Step magnitude $| \Delta x \|$")
  plt.colorbar(mappable=Q_modified, label=r"Step magnitude $| \Delta x \|$")

